# Training Loop con PyTorch

Ya construimos una red neuronal en PyTorch.

Ahora conectaremos todo lo aprendido anteriormente:

```text
Forward Pass
    ↓
Loss
    ↓
Backpropagation
    ↓
Gradient Descent
    ↓
Update Parameters
    ↓
Repeat
```

PyTorch automatiza gran parte de este proceso, pero cada línea corresponde a conceptos que ya estudiamos.

## Objetivos

Al finalizar podrás:

- definir una loss function;
- crear un optimizer;
- escribir un training loop;
- interpretar `zero_grad`, `backward` y `step`;
- visualizar la training loss;
- evaluar cómo cambian las predicciones durante el entrenamiento.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn


## 1. Preparación de los datos


In [ ]:
torch.manual_seed(42)

iris = load_iris()
X = iris.data
y = iris.target

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

X_train_t = torch.tensor(X_train, dtype=torch.float32)
X_test_t = torch.tensor(X_test, dtype=torch.float32)

y_train_t = torch.tensor(y_train, dtype=torch.long)
y_test_t = torch.tensor(y_test, dtype=torch.long)


## 2. Definimos la red


In [ ]:
model = nn.Sequential(
    nn.Linear(4, 8),
    nn.ReLU(),
    nn.Linear(8, 3)
)

print(model)


## 3. Loss function

Este es un problema de clasificación multiclase.

Usaremos:

```python
nn.CrossEntropyLoss()
```

Esta función espera:

- logits del modelo;
- labels enteras como `0`, `1`, `2`.

No necesitamos aplicar Softmax manualmente durante el entrenamiento.


In [ ]:
criterion = nn.CrossEntropyLoss()


### Pregunta

¿Por qué no aplicamos `Softmax` antes de `CrossEntropyLoss`?

<details>
<summary><strong>Pista</strong></summary>

PyTorch combina internamente las operaciones necesarias de forma numéricamente estable.

</details>

<details>
<summary><strong>Mostrar solución</strong></summary>

`CrossEntropyLoss` espera directamente los **logits**.

Aplicar Softmax manualmente antes puede ser redundante y menos estable numéricamente.

</details>


## 4. Optimizer

Usaremos Stochastic Gradient Descent:

```python
torch.optim.SGD
```

con:

\[
\eta=0.05.
\]


In [ ]:
optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.05
)


## 5. Un solo training step

Primero veremos un solo paso antes de escribir el loop completo.


In [ ]:
# 1. Limpiar gradientes anteriores
optimizer.zero_grad()

# 2. Forward pass
logits = model(X_train_t)

# 3. Loss
loss = criterion(logits, y_train_t)

print("Loss antes del backward:", loss.item())


Ahora:

```python
loss.backward()
```

calcula los gradientes.


In [ ]:
loss.backward()


Podemos inspeccionar un gradiente:


In [ ]:
for name, parameter in model.named_parameters():
    print(name, parameter.grad.shape)
    break


Finalmente:

```python
optimizer.step()
```

actualiza los parámetros.


In [ ]:
optimizer.step()


## 6. Las tres líneas fundamentales

En cada epoch:

```python
optimizer.zero_grad()
loss.backward()
optimizer.step()
```

significan:

```text
zero_grad() → limpiar gradientes anteriores

backward()  → calcular nuevos gradientes

step()      → actualizar parámetros
```


### Pregunta

¿Qué ocurriría si nunca utilizáramos `optimizer.step()`?

<details>
<summary><strong>Mostrar solución</strong></summary>

Los gradientes serían calculados, pero los weights y biases no cambiarían.

Por lo tanto, la red no aprendería.

</details>


## 7. Training loop completo

Reiniciaremos el modelo para entrenarlo desde cero.


In [ ]:
torch.manual_seed(42)

model = nn.Sequential(
    nn.Linear(4, 8),
    nn.ReLU(),
    nn.Linear(8, 3)
)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.05
)

epochs = 200
loss_history = []

for epoch in range(epochs):

    # Forward pass
    logits = model(X_train_t)

    # Loss
    loss = criterion(logits, y_train_t)

    # Backpropagation
    optimizer.zero_grad()
    loss.backward()

    # Update parameters
    optimizer.step()

    loss_history.append(loss.item())

    if epoch % 25 == 0:
        print(f"Epoch {epoch:3d} | Loss = {loss.item():.4f}")


## 8. Visualizamos la loss


In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(loss_history)
plt.xlabel("Epoch")
plt.ylabel("Cross Entropy Loss")
plt.title("Training Loss")
plt.show()


### Pregunta

¿Qué patrón esperábamos observar?

<details>
<summary><strong>Mostrar solución</strong></summary>

Esperábamos que la training loss disminuyera conforme los parámetros se ajustan.

La disminución no siempre será perfectamente suave en redes o datasets más complejos.

</details>


## 9. Accuracy durante training

Podemos medir qué proporción de ejemplos se clasifican correctamente.


In [ ]:
with torch.no_grad():
    logits = model(X_train_t)
    predictions = torch.argmax(logits, dim=1)

    train_accuracy = (predictions == y_train_t).float().mean()

print("Training accuracy =", train_accuracy.item())


## 10. Test accuracy

Ahora evaluamos datos que no participaron en el entrenamiento.


In [ ]:
with torch.no_grad():
    test_logits = model(X_test_t)
    test_predictions = torch.argmax(test_logits, dim=1)

    test_accuracy = (test_predictions == y_test_t).float().mean()

print("Test accuracy =", test_accuracy.item())


## 11. Ejercicio: cambia el learning rate

Prueba:

```python
lr = 0.005
lr = 0.05
lr = 0.5
```

y compara:

- rapidez de disminución de la loss;
- estabilidad;
- accuracy final.

<details>
<summary><strong>Pista</strong></summary>

Para una comparación justa, reinicia el modelo usando:

```python
torch.manual_seed(42)
```

antes de cada entrenamiento.

</details>

<details>
<summary><strong>Mostrar solución conceptual</strong></summary>

Un learning rate pequeño suele entrenar más lentamente.

Uno moderado puede converger más rápido.

Uno demasiado grande puede producir entrenamiento inestable o impedir que la loss disminuya correctamente.

</details>


## 12. Función de entrenamiento

Una práctica útil es encapsular el entrenamiento.


In [ ]:
def train_model(model, X, y, epochs=200, lr=0.05):

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)

    history = []

    for epoch in range(epochs):

        logits = model(X)
        loss = criterion(logits, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        history.append(loss.item())

    return history


# Para recordar

El training loop completo es:

```text
for each epoch:

    forward pass
        ↓
    calculate loss
        ↓
    zero gradients
        ↓
    backpropagation
        ↓
    update parameters
```

En PyTorch:

```python
logits = model(X)
loss = criterion(logits, y)

optimizer.zero_grad()
loss.backward()
optimizer.step()
```

Estas pocas líneas implementan todo el proceso que estudiamos matemáticamente.


## Recursos

- [PyTorch — Optimization Loop](https://docs.pytorch.org/tutorials/beginner/basics/optimization_tutorial.html)
- [PyTorch — Autograd](https://docs.pytorch.org/tutorials/beginner/basics/autogradqs_tutorial.html)
